# Historic Jeddah — External Image Inference

Use this notebook **after training** to test the fine-tuned Qwen2.5-VL model on photographs that are **outside the training and validation dataset**.

## Why keep this separate?

- Keeps the training notebook focused on reproducible training and held-out validation.
- Prevents accidental contamination of the validation set.
- Makes external/generalization testing easier to repeat.
- Lets you upload any new image without modifying the training pipeline.


In [ ]:
!pip install -q -U "transformers>=4.49.0" "accelerate>=0.34.0" "bitsandbytes>=0.43.0" "peft==0.20.0" qwen-vl-utils pillow matplotlib


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import torch
import transformers
import bitsandbytes
import peft
import matplotlib.pyplot as plt
from pathlib import Path

from google.colab import files
from PIL import Image

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
)
from peft import PeftModel
from qwen_vl_utils import process_vision_info

if not torch.cuda.is_available():
    raise RuntimeError("A GPU is recommended for external-image inference.")

print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
print("PEFT:", peft.__version__)
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)


In [ ]:
assert peft.__version__ == "0.20.0", (
    f"Expected PEFT 0.20.0, found {peft.__version__}"
)

print(f"✓ PEFT {peft.__version__}")


## 1. Model configuration


In [ ]:
BASE_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

TRAINING_ROOT = Path(
    "/content/drive/MyDrive/qwen2_5_vl_albalad_training"
)

FINAL_ADAPTER_DIR = TRAINING_ROOT / "final_lora_adapter"
PROCESSOR_DIR = TRAINING_ROOT / "processor"

MIN_PIXELS = 224 * 224
MAX_PIXELS = 224 * 224
MAX_NEW_TOKENS = 220

print(f"Adapter: {FINAL_ADAPTER_DIR}")
print(f"Processor: {PROCESSOR_DIR}")


## 2. Load the fine-tuned model

This uses the base model in 4-bit mode and attaches the saved LoRA adapter. It is usually more memory-efficient than loading a full merged model.


In [ ]:
from pathlib import Path

TRAINING_ROOT = Path(
    "/content/drive/MyDrive/qwen2_5_vl_albalad_training"
)

print("Searching for adapter_config.json...\n")

adapter_configs = list(
    TRAINING_ROOT.rglob("adapter_config.json")
)

if not adapter_configs:
    print("❌ No adapter_config.json found.")
else:
    print(f"Found {len(adapter_configs)} adapter configuration(s):\n")

    for path in adapter_configs:
        print(path)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    str(FINAL_ADAPTER_DIR),
)

fine_tuned_model.eval()

processor = AutoProcessor.from_pretrained(
    str(PROCESSOR_DIR),
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

print("Fine-tuned model loaded successfully.")


## 3. Upload an external image


In [ ]:
uploaded = files.upload()

if not uploaded:
    raise ValueError("No image was uploaded.")

uploaded_name = next(iter(uploaded))
external_image_path = Path(uploaded_name)

image = Image.open(external_image_path).convert("RGB")

print(f"Image: {external_image_path}")
print(f"Size: {image.size}")


In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.axis("off")
plt.title("External test image")
plt.show()


## 4. Enter the test instruction

Use an instruction similar to the one used during fine-tuning when you want a fair comparison.

You can change it for exploratory testing, but record the instruction when reporting results.


In [ ]:
TEST_INSTRUCTION = (
    "Describe this historic building in Al-Balad. "
    "Mention the house name if you recognize it at the start"
    "Focus on the visible architectural elements, façade, "
    "materials, openings, woodwork, and surrounding streetscape. "
    "Do not invent details that are not visible. "
    "Write 3 to 4 complete sentences in a warm tone."
)


## 5. Generate the model description


In [ ]:
def generate_external_description(
    model,
    processor,
    image,
    instruction,
    max_new_tokens=MAX_NEW_TOKENS,
):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                },
                {
                    "type": "text",
                    "text": instruction,
                },
            ],
        }
    ]

    formatted_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    model_inputs = processor(
        text=[formatted_text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    model_inputs = {
        key: value.to(model.device)
        if isinstance(value, torch.Tensor)
        else value
        for key, value in model_inputs.items()
    }

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            use_cache=True,
        )

    input_length = model_inputs["input_ids"].shape[1]

    generated_only = generated_ids[:, input_length:]

    generated_text = processor.batch_decode(
        generated_only,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    return generated_text[0].strip()

external_caption = generate_external_description(
    fine_tuned_model,
    processor,
    image,
    TEST_INSTRUCTION,
)

print("Generated architectural description:")
print()
print(external_caption)


## 6. Save the external test result

Saving the image path, instruction, and generated caption makes external testing auditable and reproducible.


In [ ]:
import json
from datetime import datetime

result = {
    "timestamp": datetime.now().isoformat(),
    "image_name": external_image_path.name,
    "instruction": TEST_INSTRUCTION,
    "generated_description": external_caption,
}

result_path = TRAINING_ROOT / "external_image_test_result.json"

with result_path.open("w", encoding="utf-8") as file:
    json.dump(result, file, ensure_ascii=False, indent=2)

print(f"Result saved to: {result_path}")


## Recommended evaluation procedure

For a proper final evaluation, keep three groups separate:

1. **Training set** — used to fit the LoRA adapter.
2. **Validation set** — used during training and for held-out model checks.
3. **External test set** — images never used during training or validation.

The external set should ideally contain different photographs, viewpoints, lighting conditions, or buildings. This gives you a much stronger measure of generalization than testing another image that was already represented in the training data.
